# 01 · Data preparation and validation

**Question.** Is the data complete and internally consistent before anything is
claimed from it?

This notebook does three things: it loads the panel for the active mode, runs the
validation checks, and prints the cause-field coverage that scopes every later
statement. No figure here — this stage exists so the next five can be trusted.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

## 1.1 Load and validate

`clean.validate` counts rows, duplicates, missingness and threshold coverage. It
raises on a duplicate rate above 0.1% or on more than 5% missing arrival delays,
because either would make the panel unusable rather than merely noisy.

In [ ]:
raw = io.load_flights()
checks = clean.validate(raw, f"panel_{config.mode()}")
pd.Series(checks, name="value").to_frame()

## 1.2 The three row populations

Mixing these up is the classic error in this dataset: cause shares describe
*delayed* flights, while means and percentiles describe *completed* flights.

In [ ]:
panel = clean.completed(raw)
universe = clean.attribution_universe(raw)

print(f"reported rows          {len(raw):>12,}")
print(f"completed flights      {len(panel):>12,}  ({len(panel)/len(raw):.1%} of reported)")
print(f"attribution universe   {len(universe):>12,}  ({len(universe)/len(panel):.1%} of completed)")

## 1.3 Coverage of the cause fields

BTS publishes the five cause-minute fields only above 15 minutes of arrival delay.
The check is direct: coverage should be 0% below the threshold and about 100% above
it, in every year.

In [ ]:
from team01 import attribution

attribution.threshold_audit(raw)

## 1.4 Per-year summary

What a reader needs before any trend line: how many flights, how late, and how
often severe.

In [ ]:
clean.summarise_panel(raw)